<a href="https://colab.research.google.com/github/JantikarShobith/BIS_LAB/blob/main/BIS_Week1_GAO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import numpy as np


DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
PERIODS_PER_DAY = 4
TOTAL_TIMESLOTS = len(DAYS) * PERIODS_PER_DAY  # 24 slots (0 to 23)

ROOMS = ["Room 101", "Room 102", "Lab A"]


CLASSES_TO_SCHEDULE = [
    ("Math 101", "Prof. Alan", "CS_Year1"),
    ("Math 101", "Prof. Alan", "CS_Year1"),
    ("Physics", "Prof. Bob", "CS_Year1"),
    ("Physics", "Prof. Bob", "CS_Year1"),
    ("Data Struct", "Prof. Charlie", "CS_Year2"),
    ("Data Struct", "Prof. Charlie", "CS_Year2"),
    ("Algorithms", "Prof. Charlie", "CS_Year2"),
    ("DB Systems", "Prof. Diana", "CS_Year2"),
    ("AI Intro", "Prof. Alan", "CS_Year3"),
    ("AI Intro", "Prof. Alan", "CS_Year3"),
    ("Networks", "Prof. Diana", "CS_Year3"),
    ("Networks", "Prof. Diana", "CS_Year3"),
]


POPULATION_SIZE = 100
MAX_GENERATIONS = 500
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.05
ELITISM_COUNT = 2

def create_random_chromosome():
    """
    A Chromosome is a list of gene dictionaries.
    Each gene represents a scheduled class: (course, prof, group, timeslot, room)
    """
    chromosome = []
    for course, prof, group in CLASSES_TO_SCHEDULE:
        timeslot = random.randint(0, TOTAL_TIMESLOTS - 1)
        room = random.choice(ROOMS)
        chromosome.append({
            'course': course,
            'prof': prof,
            'group': group,
            'timeslot': timeslot,
            'room': room
        })
    return chromosome

def evaluate_fitness(chromosome):
    """
    Calculates fitness based on penalty reduction.
    Penalty increases for every hard-constraint violation (collisions).
    Fitness = 1.0 / (1.0 + total_penalties) -> Max fitness is 1.0 (0 clashes).
    """
    penalties = 0
    num_classes = len(chromosome)

    for i in range(num_classes):
        for j in range(i + 1, num_classes):
            c1 = chromosome[i]
            c2 = chromosome[j]

            # Hard Constraint 1: Same Timeslot Collisions
            if c1['timeslot'] == c2['timeslot']:
                # Same Professor double-booked
                if c1['prof'] == c2['prof']:
                    penalties += 1
                # Same Student Group double-booked
                if c1['group'] == c2['group']:
                    penalties += 1
                # Same Room double-booked
                if c1['room'] == c2['room']:
                    penalties += 1

    # Fitness equation: perfect score is 1.0
    return 1.0 / (1.0 + penalties), penalties

def selection(population, fitnesses):
    """ Tournament Selection """
    tournament_size = 3
    selected = random.sample(list(zip(population, fitnesses)), tournament_size)
    selected.sort(key=lambda x: x[1], reverse=True)
    return selected[0][0]

def crossover(parent1, parent2):
    """ Single-point Crossover on class assignments """
    if random.random() > CROSSOVER_RATE:
        return [dict(g) for g in parent1], [dict(g) for g in parent2]

    crossover_point = random.randint(1, len(CLASSES_TO_SCHEDULE) - 1)

    child1 = [dict(g) for g in parent1[:crossover_point]] + [dict(g) for g in parent2[crossover_point:]]
    child2 = [dict(g) for g in parent2[:crossover_point]] + [dict(g) for g in parent1[crossover_point:]]

    return child1, child2

def mutate(chromosome):
    """ Mutates a schedule by reassignment of timeslot or room """
    for gene in chromosome:
        if random.random() < MUTATION_RATE:
            gene['timeslot'] = random.randint(0, TOTAL_TIMESLOTS - 1)
        if random.random() < MUTATION_RATE:
            gene['room'] = random.choice(ROOMS)
    return chromosome

# ---------------------------------------------------------
# 4. Main Genetic Algorithm Loop
# ---------------------------------------------------------
def run_timetable_ga():
    random.seed(42)

    # 1. Initialize Population
    population = [create_random_chromosome() for _ in range(POPULATION_SIZE)]

    print("Beginning Timetable Evolutionary Optimization...\n")

    for generation in range(MAX_GENERATIONS):
        # Evaluate Fitness
        fitness_results = [evaluate_fitness(chrom) for chrom in population]
        fitnesses = [fit for fit, _ in fitness_results]
        penalties = [pen for _, pen in fitness_results]

        best_idx = np.argmax(fitnesses)
        best_fitness = fitnesses[best_idx]
        best_penalty = penalties[best_idx]

        if generation % 50 == 0 or best_penalty == 0:
            print(f"Gen {generation:3d} | Best Fitness: {best_fitness:.4f} | Total Clashes: {best_penalty}")

        # Termination condition: Perfect schedule found
        if best_penalty == 0:
            print(f"\nSUCCESS: Conflict-free timetable found in Generation {generation}!")
            return population[best_idx]

        # 2. Elitism: Preserve best individuals directly
        sorted_pop = [x for _, x in sorted(zip(fitnesses, population), key=lambda pair: pair[0], reverse=True)]
        new_population = [ [dict(g) for g in chrom] for chrom in sorted_pop[:ELITISM_COUNT] ]

        # 3. Create next generation through Selection, Crossover, Mutation
        while len(new_population) < POPULATION_SIZE:
            p1 = selection(population, fitnesses)
            p2 = selection(population, fitnesses)

            c1, c2 = crossover(p1, p2)

            new_population.append(mutate(c1))
            if len(new_population) < POPULATION_SIZE:
                new_population.append(mutate(c2))

        population = new_population

    # Return best available if max gen reached
    fitness_results = [evaluate_fitness(chrom) for chrom in population]
    best_idx = np.argmax([fit for fit, _ in fitness_results])
    return population[best_idx]

# ---------------------------------------------------------
# 5. Output Formatting
# ---------------------------------------------------------
def print_timetable(schedule):
    print("\n" + "="*80)
    print(f"{'6-DAY GENERATED CLASSROOM TIMETABLE':^80}")
    print("="*80)

    # Organize schedule by day and period
    timetable_grid = {day: {period: [] for period in range(1, PERIODS_PER_DAY + 1)} for day in DAYS}

    for gene in schedule:
        day_idx = gene['timeslot'] // PERIODS_PER_DAY
        period_idx = (gene['timeslot'] % PERIODS_PER_DAY) + 1
        day_name = DAYS[day_idx]

        info = f"{gene['course']} | {gene['prof']} | {gene['group']} | [{gene['room']}]"
        timetable_grid[day_name][period_idx].append(info)

    for day in DAYS:
        print(f"\n--- {day.upper()} ---")
        for period in range(1, PERIODS_PER_DAY + 1):
            classes = timetable_grid[day][period]
            if classes:
                for cls in classes:
                    print(f"  Period {period}: {cls}")
            else:
                print(f"  Period {period}: [FREE PERIOD]")

# Execute the algorithm
best_schedule = run_timetable_ga()
print_timetable(best_schedule)

Beginning Timetable Evolutionary Optimization...

Gen   0 | Best Fitness: 1.0000 | Total Clashes: 0

SUCCESS: Conflict-free timetable found in Generation 0!

                      6-DAY GENERATED CLASSROOM TIMETABLE                       

--- MONDAY ---
  Period 1: Math 101 | Prof. Alan | CS_Year1 | [Lab A]
  Period 2: AI Intro | Prof. Alan | CS_Year3 | [Room 101]
  Period 3: AI Intro | Prof. Alan | CS_Year3 | [Room 101]
  Period 4: [FREE PERIOD]

--- TUESDAY ---
  Period 1: [FREE PERIOD]
  Period 2: [FREE PERIOD]
  Period 3: [FREE PERIOD]
  Period 4: Physics | Prof. Bob | CS_Year1 | [Room 101]
  Period 4: Networks | Prof. Diana | CS_Year3 | [Lab A]

--- WEDNESDAY ---
  Period 1: Physics | Prof. Bob | CS_Year1 | [Room 101]
  Period 2: [FREE PERIOD]
  Period 3: [FREE PERIOD]
  Period 4: [FREE PERIOD]

--- THURSDAY ---
  Period 1: [FREE PERIOD]
  Period 2: [FREE PERIOD]
  Period 3: [FREE PERIOD]
  Period 4: [FREE PERIOD]

--- FRIDAY ---
  Period 1: [FREE PERIOD]
  Period 2: Algorithms |